# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [44]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import joblib
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from tqdm.notebook import tqdm

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [26]:
df = pd.read_csv("../data/day-of-week-not-scaled.csv")
df_days = pd.read_csv("../data/dayofweek.csv")
df["dayofweek"] = df_days["dayofweek"].values

X = df.drop(columns="dayofweek")
y = df["dayofweek"]

Первый шаг (Выделяем чистый тест): Вы берете исходные X и y и отрезаете от них 
данных. Получаются X_test и y_test (их мы откладываем в сторону до самого финала). Оставшиеся 
временно называются X_train и y_train.
Второй шаг (Выделяем валидацию): Вы берете полученные на первом шаге X_train и y_train (
от исходного датасета) и снова отправляете их в train_test_split. От них отрезается еще кусочек (обычно тоже), который перезаписывается в X_valid и y_valid, а остаток окончательно становится вашим рабочим X_train и y_train.


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, random_state=21, stratify=y_train
)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [28]:
svc_best_params = {
    "C": 10,
    "class_weight": None,
    "gamma": "auto",
    "kernel": "rbf",
    "probability": True,
    "random_state": 21,
}

dt_best_params = {
    "class_weight": "balanced",
    "criterion": "gini",
    "max_depth": 21,
    "random_state": 21,
}

rf_best_params = {
    "class_weight": "balanced",
    "criterion": "entropy",
    "max_depth": 24,
    "n_estimators": 100,
    "random_state": 21,
}

In [29]:
svc_model = SVC(**svc_best_params).fit(X_train, y_train)
dt_model = DecisionTreeClassifier(**dt_best_params).fit(X_train, y_train)
rf_model = RandomForestClassifier(**rf_best_params).fit(X_train, y_train)

In [30]:
y_pred_svc = svc_model.predict(X_valid)
y_pred_dt = dt_model.predict(X_valid)
y_pred_rf = rf_model.predict(X_valid)

In [31]:
for model_name, y_pred in zip(("svc", "dt", "rf"), (y_pred_svc, y_pred_dt, y_pred_rf)):
    acc = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_valid, y_pred, average="weighted")
    print(f"Метрики для {model_name}:")
    print(f"accuracy is {acc:.5f}")
    print(f"precision is {precision:.5f}")
    print(f"recall is {recall:.5f}\n")

Метрики для svc:
accuracy is 0.87778
precision is 0.88162
recall is 0.87778

Метрики для dt:
accuracy is 0.86667
precision is 0.87170
recall is 0.86667

Метрики для rf:
accuracy is 0.89630
precision is 0.89698
recall is 0.89630



## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.

In [32]:
# по дефолту voting='hard' (голосование большинством голосовесли две модели сказали «Понедельник», а одна «Вторник», ансамбль выберет Понедельник)
vcl = VotingClassifier(
    estimators=[("svc", svc_model), ("dt", dt_model), ("rf", rf_model)],
).fit(X_train, y_train)

y_pred_vcl = vcl.predict(X_valid)

acc_vcl = accuracy_score(y_valid, y_pred_vcl)
precision_vcl = precision_score(
    y_valid, y_pred_vcl, average="weighted", zero_division=0
)
recall_vcl = recall_score(y_valid, y_pred_vcl, average="weighted")
print(f"accuracy is {acc_vcl:.5f}")
print(f"precision is {precision_vcl:.5f}")
print(f"recall is {recall_vcl:.5f}\n")

accuracy is 0.90000
precision is 0.89993
recall is 0.90000



2. Play with the other parameteres.


In [51]:
# меняю voting='soft' (мягкое голосование по средневзвешенным вероятностям классов)
weight_range = range(1, 6)
models = []

for w_svc in tqdm(weight_range):
    for w_dt in tqdm(weight_range, leave=False):
        for w_rf in tqdm(weight_range, leave=False):
            vcl = VotingClassifier(
                estimators=[("svc", svc_model), ("dt", dt_model), ("rf", rf_model)],
                voting="soft",  # Веса лучше всего работают именно в мягком голосовании
                weights=(
                    w_svc,
                    w_dt,
                    w_rf,
                ),  # Порядок строго совпадает сestimators: svc=1, dt=1, rf=3
                n_jobs=-1,
            ).fit(X_train, y_train)
            y_pred_vcl = vcl.predict(X_valid)

            acc_vcl = accuracy_score(y_valid, y_pred_vcl)
            precision_vcl = precision_score(
                y_valid, y_pred_vcl, average="weighted", zero_division=0
            )
            recall_vcl = recall_score(y_valid, y_pred_vcl, average="weighted")

            models.append(
                {
                    "weights": vcl.get_params()["weights"],
                    "accuracy": acc_vcl,
                    "precision": precision_vcl,
                    "recall": recall_vcl,
                }
            )

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

... (21 outputs left out)

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

In [34]:
vcl_df = pd.DataFrame(models).sort_values(
    by=["accuracy", "precision"], ascending=[False, False]
)

vcl_df.head(1)

,weights,accuracy,precision,recall
78,"(4, 1, 4)",0.911111,0.912881,0.911111


In [35]:
# лучшие веса из первой строчки любого датафрема
best_weights = vcl_df.iloc[0]["weights"]
svc, dt, rf = best_weights
print(f"Веса моделей: svc = {svc}, dt = {dt}, rf = {rf}")

Веса моделей: svc = 4, dt = 1, rf = 4


3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [36]:
vcl_final = VotingClassifier(
    estimators=[("svc", svc_model), ("dt", dt_model), ("rf", rf_model)],
    voting="soft",
    weights=best_weights,
).fit(X_train, y_train)
y_pred_vcl = vcl.predict(X_valid)

acc_vcl_fin = accuracy_score(y_valid, y_pred_vcl)
precision_vcl_fin = precision_score(
    y_valid, y_pred_vcl, average="weighted", zero_division=0
)
recall_vcl_fin = recall_score(y_valid, y_pred_vcl, average="weighted")
print(f"accuracy is {acc_vcl_fin:.5f}")
print(f"precision is {precision_vcl_fin:.5f}")
print(f"recall is {recall_vcl_fin:.5f}\n")

accuracy is 0.88519
precision is 0.88840
recall is 0.88519



## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.

In [ ]:
bclf_models = []

for est in tqdm(range(10, 100, 10)):
    bclf = BaggingClassifier(
        base_estimator=svc_model,
        n_estimators=est,
        random_state=21,
        n_jobs=-1,
    ).fit(X_train, y_train)
    y_pred_bclf = bclf.predict(X_valid)

    acc_bclf = accuracy_score(y_valid, y_pred_bclf)
    precision_bclf = precision_score(
        y_valid, y_pred_bclf, average="weighted", zero_division=0
    )
    recall_bclf = recall_score(y_valid, y_pred_bclf, average="weighted")
    bclf_models.append(
        {
            "est": est,
            "acc": acc_bclf,
            "precision": precision_bclf,
            "recall": recall_bclf,
        }
    )

  0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
bclf_df = (
    pd.DataFrame(bclf_models)
    .sort_values(by=["acc", "precision"], ascending=[False, False])
    .head(3)
)

print("на первом шаге видим, что лидер модель с n_estimators=30:")
bclf_df.style.highlight_max(
    subset=["acc", "precision"], color="lightgreen", axis=0
).hide_index()

на первом шаге видим, что лидер модель с n_estimators=30:


est,acc,precision,recall
30,0.888889,0.897182,0.888889
10,0.885185,0.894269,0.885185
70,0.885185,0.893961,0.885185


2. Play with the other parameters.

In [117]:
bclf_models2 = []

for sample_ratio in tqdm([0.7, 0.8, 0.9, 1.0]):
    bclf_tuned = BaggingClassifier(
        base_estimator=svc_model,
        n_estimators=30,
        max_samples=sample_ratio,
        random_state=21,
        n_jobs=-1,
    ).fit(X_train, y_train)

    y_pred_tuned = bclf_tuned.predict(X_valid)

    acc_t = accuracy_score(y_valid, y_pred_tuned)
    precision_t = precision_score(
        y_valid, y_pred_tuned, average="weighted", zero_division=0
    )

    bclf_models2.append(
        {
            "est": 30,
            "max_samples": sample_ratio,
            "acc": acc_t,
            "precision": precision_t,
        }
    )

  0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
bclf_df2 = pd.DataFrame(bclf_models2).sort_values(
    by=["acc", "precision"], ascending=[False, False]
)
print("""Игра с параметрами, а именно уменьшение max_samples (по умолчанию=1) не улучшило рез-ты модели,
что логично, поскольку модель учится не на всей выборке, а только на случайных 90, 80, 70% строк""")
# bclf_df2.head(10)
bclf_df2.style.highlight_max(
    subset=["acc", "precision"], color="lightgreen", axis=0
).format({"max_samples": "{:.1f}"}).hide_index()

Игра с параметрами, а именно уменьшение max_samples (по умолчанию=1) не улучшило рез-ты модели,
что логично, поскольку модель учится не на всей выборке, а только на случайных 90, 80, 70% строк


est,max_samples,acc,precision
30,1.0,0.888889,0.897182
30,0.9,0.877778,0.886915
30,0.8,0.870370,0.880927
30,0.7,0.862963,0.873096


3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [ ]:
final_models = []

for final_est in [30, 40, 50, 60]:
    bclf_final = BaggingClassifier(
        base_estimator=svc_model,
        n_estimators=final_est,
        random_state=21,
        n_jobs=-1,
    ).fit(X_train, y_train)

    y_pred_final_test = bclf_final.predict(X_test)
    acc_f = accuracy_score(y_test, y_pred_final_test)
    prec_f = precision_score(
        y_test, y_pred_final_test, average="weighted", zero_division=0
    )
    rec_f = recall_score(y_test, y_pred_final_test, average="weighted")

    final_models.append(
        {"est": final_est, "acc": acc_f, "precision": prec_f, "rec_f": rec_f}
    )

In [ ]:
bclf_fin = pd.DataFrame(final_models).sort_values(
    by=["acc", "precision"], ascending=[False, False]
)
bclf_fin.style.highlight_max(
    subset=["acc", "precision"], color="lightgreen", axis=0
).format({"max_samples": "{:.1f}"}).hide_index()

est,acc,precision,rec_f
50,0.884615,0.889413,0.884615
60,0.878698,0.884291,0.878698
40,0.875740,0.881158,0.875740
30,0.872781,0.878404,0.872781


In [ ]:
print(f"""Таким образом, несмотря на то, что на валидационной выборке лидером была модель с n_estimators={bclf_df2["est"].iloc[0]}, 
при расчете метрик на тестовой выборке лидером стала модель с n_estimators={bclf_fin["est"].iloc[0]}. Валидационная выборка 
содержит {len(X_valid)} строк, тестовая {len(X_test)} строк. Метрики на тестовой выборке показывают, что чем больше базовых 
моделей в ансамбле, тем стабильнее и сильнее он становится на новых, незнакомых данных, даже если на 
маленьком валидационном кусочке метрики временно колебались. 
Модель из 50 базовых SVM гораздо лучше обобщает данные, чем модель из 30.""")

Таким образом, несмотря на то, что на валидационной выборке лидером была модель с n_estimators=30, 
при расчете метрик на тестовой выборке лидером стала модель с n_estimators=50. Валидационная выборка 
содержит 270 строк, тестовая 338 строк. Метрики на тестовой выборке показывают, что чем больше базовых 
моделей в ансамбле, тем стабильнее и сильнее он становится на новых, незнакомых данных, даже если на 
маленьком валидационном кусочке метрики временно колебались. 
Модель из 50 базовых SVM гораздо лучше обобщает данные, чем модель из 30.


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.